In [2]:
import warnings
warnings.filterwarnings('ignore')

# Core
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistics
from scipy import stats
from scipy.stats import chi2_contingency
import statsmodels.api as sm
from statsmodels.formula.api import logit

# ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
import xgboost as xgb
import lightgbm as lgb

# Survival analysis
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.plotting import add_at_risk_counts

# Time-series
from prophet import Prophet          # uncomment if installed
from pmdarima import auto_arima      # uncomment if installed

#Geo (optional — comment out if geopandas not installed)
import geopandas as gpd
import folium

# Global plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})


In [3]:
import pandas as pd
df = pd.read_csv("df_clean.csv")

# ════════════════════════════════════════════════════════════════════════════════
# SECTION 10 — ADVANCED STATISTICAL & MACHINE LEARNING ANALYSIS
# ════════════════════════════════════════════════════════════════════════════════

In [4]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Optional libraries (install if needed)
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print(" XGBoost not installed – skipping XGBoost models.")

try:
    import statsmodels.api as sm
    HAS_STATS = True
except ImportError:
    HAS_STATS = False
    print(" statsmodels not installed – skipping odds ratio analysis.")

from sklearn.inspection import permutation_importance

# Set style
sns.set_style("whitegrid")

In [9]:
missing_df = missing.reset_index()
missing_df.columns = ['column', 'missing_count']
missing_df

,column,missing_count
0,bmi_cat_end,8540
1,bmi_cat_start,8540
2,date_mdr_interim,8510
3,treatment_days,8385
4,date_cotrimoxazole_start,8072
5,diag_delay_days,7761
6,date_art_start,7546
7,date_treatment_start,7499
8,date_notified,7499
9,date_smear_result,7063


In [10]:
cols_to_drop = [
    'bmi_cat_end',
    'bmi_cat_start',
    'date_mdr_interim'
]

df = df.drop(columns=cols_to_drop)

In [14]:
df.head()

,facility,date_notified,year,month,fy,district,bact_confirmed,tb_type,site_of_disease,drug_sensitivity,...,treatment_days,weight_change_kg,outcome_success,died,ltfu,tx_failed,is_dr,hiv_pos,tpt_started_total,tx_start_is_proxy
0,Ruhengeri RH,2024-04-02 00:00:00.000,2024.0,4.0,2024.0,Musanze District,Clinically diagnosed,Pleural TB,Extra pulmonary,DS-TB,...,NaN,-58.0,0,0,0,0,0,0,0,True
1,Kicukiro CS,2024-03-05 00:00:00.000,2024.0,3.0,2023.0,Kicukiro District,Bacteriologically confirmed,Unknown,Pulmonary,DS-TB,...,NaN,-54.0,0,0,0,0,0,1,0,True
2,Kairos CS,2024-02-02 00:00:00.000,2024.0,2.0,2023.0,Kicukiro District,Bacteriologically confirmed,Unknown,Pulmonary,DS-TB,...,NaN,-45.0,0,0,0,0,0,0,0,True
3,Kicukiro CS,2024-03-15 00:00:00.000,2024.0,3.0,2023.0,Kicukiro District,Bacteriologically confirmed,Unknown,Pulmonary,DS-TB,...,NaN,-50.0,0,0,0,0,0,0,0,True
4,Rubavu Prison,2024-04-05 00:00:00.000,2024.0,4.0,2024.0,Rubavu District,Bacteriologically confirmed,Unknown,Pulmonary,DS-TB,...,NaN,-52.0,0,0,0,0,0,0,0,True


In [16]:
df['date_control_end'] = pd.to_datetime(
    df['date_control_end'], errors='coerce', format='mixed'
)

df['date_treatment_start'] = pd.to_datetime(
    df['date_treatment_start'], errors='coerce', format='mixed'
)

In [19]:
df['treatment_days'] = (
    df['date_control_end'] - df['date_treatment_start']
).dt.days

In [18]:
df.loc[df['treatment_days'] < 0, 'treatment_days'] = None

In [21]:
df[df['date_control_end'].isnull()][['date_control_end']].head()

,date_control_end
0,NaT
1,NaT
2,NaT
3,NaT
4,NaT


In [22]:
df['treatment_completed'] = df['date_control_end'].notnull().astype(int)

In [23]:
df['treatment_days'] = (
    df['date_control_end'] - df['date_treatment_start']
).dt.days

In [24]:
df['treatment_days'] = df['treatment_days'].fillna(0)

In [26]:
df.isna().sum()

facility                  0
date_notified          7499
year                   1765
month                  1765
fy                     1765
                       ... 
is_dr                     0
hiv_pos                   0
tpt_started_total         0
tx_start_is_proxy         0
treatment_completed       0
Length: 106, dtype: int64

In [32]:
missing_df = pd.DataFrame(missing_list, columns=['column', 'missing_percent'])

# Round for readability
missing_df['missing_percent'] = missing_df['missing_percent'].round(2)

missing_df

,column,missing_percent
0,date_cotrimoxazole_start,94.52
1,diag_delay_days,90.88
2,date_art_start,88.36
3,date_notified,87.81
4,date_treatment_start,87.81
...,...,...
101,is_dr,0.00
102,hiv_pos,0.00
103,tpt_started_total,0.00
104,tx_start_is_proxy,0.00


In [35]:
pd.set_option('display.max_rows', None)
missing_df

,column,missing_percent
0,date_cotrimoxazole_start,94.52
1,diag_delay_days,90.88
2,date_art_start,88.36
3,date_notified,87.81
4,date_treatment_start,87.81
5,date_smear_result,82.70
6,date_control_end,68.03
7,date_control_m5,65.53
8,date_control_m2,46.32
9,date_xpert_result,23.75


In [38]:
engineer_cols = [
    'date_control_end',
    'date_control_m5',
    'date_control_m2'
]

In [39]:
df['treatment_completed'] = df['date_control_end'].notnull().astype(int)
df['control_m5_done'] = df['date_control_m5'].notnull().astype(int)
df['control_m2_done'] = df['date_control_m2'].notnull().astype(int)

In [40]:
for col in ['fy', 'quarter', 'month', 'year']:
    df[col].fillna(df[col].mode()[0], inplace=True)

In [42]:
df['treatment_completed'] = df['date_control_end'].notnull().astype(int)
df['control_m5_done'] = df['date_control_m5'].notnull().astype(int)
df['control_m2_done'] = df['date_control_m2'].notnull().astype(int)

In [43]:
df['has_xpert_result'] = df['date_xpert_result'].notnull().astype(int)
df['has_xpert_collected'] = df['date_xpert_collected'].notnull().astype(int)

In [ ]:
df = df.drop(columns=[
    'date_control_end',
    'date_control_m5',
    'date_control_m2',
    'date_xpert_result',
    'date_xpert_collected'
])

In [49]:
df.isna().sum().sum()

np.int64(45440)

In [47]:
df.shape

(8540, 105)

In [50]:
df.fillna(df.median(numeric_only=True), inplace=True)

In [51]:
df.isna().sum().sum()

np.int64(37679)

In [52]:
int(df.isna().sum().sum())

37679

In [53]:
# DO NOT over-impute
# Keep missing values

X = df.drop(columns=['died'])
y = df['died']

from xgboost import XGBClassifier

model = XGBClassifier()
model.fit(X, y)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:facility: object, date_notified: object, district: object, bact_confirmed: object, tb_type: object, site_of_disease: object, drug_sensitivity: object, tb_category: object, xpert_result: object, xpert_rifampicin: object, smear_result: object, date_smear_result: object, who_category: object, mwrd: object, dst_result: object, culture_result: object, tb_lam_test: object, tb_lam_result: object, hiv_status: object, hiv_history: object, on_cotrimoxazole: object, date_cotrimoxazole_start: object, on_art: object, date_art_start: object, sex: object, dob: object, age_cat: object, age_group: object, hrg_cat: object, hrg: object, referred_by: object, contact_tb_pos: object, contact_mdr_tb: object, diabetic: object, health_worker: object, chw: object, miner: object, prisoners: object, refugee: ob